# Assigment 4

In [29]:
import numpy as np
import scipy.sparse as sp

from math import sqrt

import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [30]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

def plot_lines(p, v0, v1, color):
    tmp = np.zeros((v0.shape[0] * 2, 3))
    tmp[0::2] = v0
    tmp[1::2] = v1
    p.add_lines(tmp, color=color)

In [31]:
v, f = igl.read_triangle_mesh("data/irr4-cyl2.off")
tt, _ = igl.triangle_triangle_adjacency(f)

c = np.loadtxt("data/irr4-cyl2.constraints")
cf = c[:, 0].astype(np.int64)
c = c[:, 1:]

In [32]:
def align_field(V, F, TT, soft_id, soft_value, llambda):
    assert(soft_id[0] > 0)
    assert(soft_id.shape[0] == soft_value.shape[0])


    # Edges
    e1 = V[F[:, 1], :] - V[F[:, 0], :]
    e2 = V[F[:, 2], :] - V[F[:, 0], :]

    # Compute the local reference systems for each face, T1, T2
    T1 = e1 / np.linalg.norm(e1, axis=1)[:,None]

    T2 =  np.cross(T1, np.cross(T1, e2))
    T2 /= np.linalg.norm(T2, axis=1)[:,None]

    # Arrays for the entries of the matrix
    data = []
    ii = []
    jj = []

    index = 0
    for f in range(F.shape[0]):
        for ei in range(3): # Loop over the edges

            # Look up the opposite face
            g = TT[f, ei]

            # If it is a boundary edge, it does not contribute to the energy
            # or avoid to count every edge twice
            if g == -1 or f > g:
                continue

            # Compute the complex representation of the common edge
            e  = V[F[f, (ei+1)%3], :] - V[F[f, ei], :]

            vef = np.array([np.dot(e, T1[f, :]), np.dot(e, T2[f, :])])
            vef /= np.linalg.norm(vef)
            ef = (vef[0] + vef[1]*1j).conjugate()

            veg = np.array([np.dot(e, T1[g, :]), np.dot(e, T2[g, :])])
            veg /= np.linalg.norm(veg)
            eg = (veg[0] + veg[1]*1j).conjugate()


            # Add the term conj(f)^n*ui - conj(g)^n*uj to the energy matrix
            data.append(ef);  ii.append(index); jj.append(f)
            data.append(-eg); ii.append(index); jj.append(g)

            index += 1


    sqrtl = sqrt(llambda)

    # Convert the constraints into the complex polynomial coefficients and add them as soft constraints

    # Rhs of the system
    b = np.zeros(index + soft_id.shape[0], dtype=complex)

    for ci in range(soft_id.shape[0]):
        f = soft_id[ci]
        v = soft_value[ci, :]

        # Project on the local frame
        c = np.dot(v, T1[f, :]) + np.dot(v, T2[f, :])*1j

        data.append(sqrtl); ii.append(index); jj.append(f)
        b[index] = c * sqrtl

        index += 1

    assert(b.shape[0] == index)


    # Solve the linear system
    A = sp.coo_matrix((data, (ii, jj)), shape=(index, F.shape[0])).asformat("csr")
    u = sp.linalg.spsolve(A.conjugate().T @ A, A.conjugate().T @ b)

    R = T1 * u.real[:,None] + T2 * u.imag[:,None]

    return R

In [33]:
def plot_mesh_field(V, F, R, constrain_faces):
    col = np.ones(f.shape[0])
    col[constrain_faces] = 0

    avg = igl.avg_edge_length(V, F)/2

    #Plot from face barycenters
    B = igl.barycenter(V, F)

    p = pv.Plotter()
    p.add_mesh(to_pyvista_mesh(V, F), scalars=col, show_edges=True)
    plot_lines(p, B, B + R * avg, "black")

    return p

In [ ]:
R = align_field(v, f, tt, cf, c, 1e6)
plot_mesh_field(v, f, R, cf).show()

In [ ]:
print(c)
print(cf)

In [ ]:
def reconstruct_scalar_from_face_field(V, F, U, anchor=0, anchor_value=0.0):
    G = igl.grad(V, F).tocsr() 

    A = 0.5 * igl.doublearea(V, F) 
    W = sp.diags(np.tile(A, 3))    

    u = U.T.reshape(-1, 1)

    # Normal system
    L = (G.T @ W @ G).tocsr()
    rhs = G.T @ W @ u

    # Fix one vertex to remove constant nullspace
    n = V.shape[0]
    I = np.arange(n)
    free = I[I != anchor]

    Lff = L[free][:, free]
    rhs_f = np.asarray(rhs[free]).ravel() - L[free, anchor].toarray().ravel() * anchor_value

    s = np.zeros((n, 1))
    s[anchor, 0] = anchor_value
    s[free, 0] = sp.linalg.spsolve(Lff.tocsc(), rhs_f)

    # Reconstructed gradient per face (F x 3)
    g = (G @ s).reshape(3, F.shape[0]).T
    return s, g, A

In [ ]:
# Reconstruct scalar field from your current vector field R
s, g_rec, A = reconstruct_scalar_from_face_field(v, f, R, anchor=0, anchor_value=0.0)

# Error per face
err = np.linalg.norm(g_rec - R, axis=1)

# Visualize scalar field + reconstructed gradient and error map
B = igl.barycenter(v, f)
avg = igl.avg_edge_length(v, f) / 2
grad_magnitude = np.linalg.norm(g_rec, axis=1)

# Create meshes for visualization
m_scalar = to_pyvista_mesh(v, f)
m_scalar.point_data["s"] = s.ravel()

m_grad_mag = to_pyvista_mesh(v, f)
m_grad_mag.cell_data["grad_magnitude"] = grad_magnitude

m_error = to_pyvista_mesh(v, f)
m_error.cell_data["err"] = err

p = pv.Plotter(shape=(1, 2), window_size=(1400, 1000))

p.subplot(0, 0)
p.add_text("Reconstructed Scalar Field", font_size=11)
p.add_mesh(m_scalar, scalars="s", cmap="viridis", show_edges=False)
plot_lines(p, B, B + g_rec * avg, "black")

p.subplot(0, 1)
p.add_text("Poisson Reconstruction Error", font_size=11)
p.add_mesh(m_error, scalars="err", cmap="magma", show_edges=False)
plot_lines(p, B, B + R * avg, "white")

p.show()

### Harmonic and LSCM Parameterizations

In [ ]:
v, f = igl.read_triangle_mesh("data/camel_head.off")
v = v.astype(np.float64)
f = f.astype(np.int32)

# Helpers
def compute_gradient(verts, faces, scalar_field):
    G = igl.grad(np.asarray(verts, dtype=np.float64), faces)   # (3F x V)
    g = G @ np.asarray(scalar_field, dtype=np.float64)         # (3F,)
    nf = faces.shape[0]
    return np.column_stack([g[0:nf], g[nf:2*nf], g[2*nf:3*nf]])  # (F x 3)

def as_polydata(verts, faces):
    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).ravel()
    return pv.PolyData(verts, faces_pv)

def unpack_lscm(out, nverts):
    if isinstance(out, tuple) and len(out) == 2:
        a0 = np.asarray(out[0])
        a1 = np.asarray(out[1])
        if a0.ndim == 2 and a0.shape[1] == 2:
            uv = a0
        elif a1.ndim == 2 and a1.shape[1] == 2:
            uv = a1
    else:
        uv = np.asarray(out)

    if uv.shape == (2, nverts):
        uv = uv.T
    return uv

def add_scalar_on_surface(plotter, r, c, mesh, scalar, title, cmap):
    m = mesh.copy()
    m.point_data["phi"] = scalar
    clim = np.percentile(scalar, [2, 98]) 
    plotter.subplot(r, c)
    plotter.add_text(title, font_size=10)
    plotter.add_mesh(m, scalars="phi", cmap=cmap, clim=clim, show_edges=False, smooth_shading=True, ambient=0.25, diffuse=0.7, specular=0.1)

def add_uv_gradmag(plotter, r, c, uv, faces, scalar, title):
    uv3 = np.column_stack([uv, np.zeros(uv.shape[0])])
    m = as_polydata(uv3, faces)
    grad_uv = compute_gradient(uv3, faces, scalar)
    m.cell_data["grad_mag"] = np.linalg.norm(grad_uv, axis=1)

    plotter.subplot(r, c)
    plotter.add_text(title, font_size=10)
    plotter.add_mesh(m, scalars="grad_mag", cmap="viridis", show_edges=True, line_width=0.25, show_scalar_bar=False)
    plotter.view_xy()

# Parameterizations
bnd = igl.boundary_loop(f)
bnd_uv = igl.map_vertices_to_circle(v, bnd)
uv_h = np.asarray(igl.harmonic(v, f, bnd, bnd_uv, 1))

b = np.array([bnd[0], bnd[len(bnd) // 2]], dtype=np.int64)
bc = np.array([[0.0, 0.0], [1.0, 0.0]], dtype=np.float64)
uv_l = unpack_lscm(igl.lscm(v, f, b, bc), v.shape[0])

phi_h = uv_h[:, 0]
phi_l = uv_l[:, 0]

# Plot
mesh_3d = as_polydata(v, f)
pl = pv.Plotter(shape=(2, 2), window_size=(1700, 950))
pl.set_background("white")

add_scalar_on_surface(pl, 0, 0, mesh_3d, phi_h, f"Harmonic U on 3D", "turbo")
add_uv_gradmag(pl, 0, 1, uv_h, f, phi_h, f"Harmonic grad U in UV")

add_scalar_on_surface(pl, 1, 0, mesh_3d, phi_l, f"LSCM U on 3D", "turbo")
add_uv_gradmag(pl, 1, 1, uv_l, f, phi_l, f"LSCM grad U in UV")

pl.show()

### Editing a parameterization with vector fields

In [ ]:
# Helpers
def tri_polydata(verts, faces):
    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3, dtype=np.int64), faces]).ravel()
    return pv.PolyData(verts, faces_pv)

def signed_double_area_uv(UV, F):
    u0 = UV[F[:, 0], 0]
    v0 = UV[F[:, 0], 1]
    u1 = UV[F[:, 1], 0]
    v1 = UV[F[:, 1], 1]
    u2 = UV[F[:, 2], 0]
    v2 = UV[F[:, 2], 1]
    return (u1 - u0) * (v2 - v0) - (u2 - u0) * (v1 - v0)


def local_flip_mask(UV, F, eps=1e-14):
    """Faces with orientation opposite to the map's global orientation (or nearly degenerate)."""
    a = signed_double_area_uv(UV, F)
    total = np.sum(a)
    global_sign = 1.0 if total >= 0.0 else -1.0
    return (global_sign * a <= eps)

# Not sure if there was supposed to be another contraint file yeilding a diff harmonic parameterization/grad
# so I just used the provided irr4-cyl2 .off and constraints to replace U coordinate in the harmonic map, which just gave the same grad as the reconstructed scalar field one.

V, F = igl.read_triangle_mesh("data/irr4-cyl2.off")
V = V.astype(np.float64)
F = F.astype(np.int32)
TT, _ = igl.triangle_triangle_adjacency(F)

C = np.loadtxt("data/irr4-cyl2.constraints")
c_faces = C[:, 0].astype(np.int64)
c_vecs = C[:, 1:].astype(np.float64)

# Initial harmonic parameterization
bnd = igl.boundary_loop(F)
bnd_uv = igl.map_vertices_to_circle(V, bnd)
UV0 = np.asarray(igl.harmonic(V, F, bnd, bnd_uv, 1), dtype=np.float64)

# Build smooth guidance field and reconstruct scalar
R = align_field(V, F, TT, c_faces, c_vecs, 1e6)
s, g_rec, A = reconstruct_scalar_from_face_field(V, F, R, anchor=0, anchor_value=0.0)
s = s.ravel()

# Replace U coordinate in harmonic map
u0 = UV0[:, 0]
s01 = (s - s.min()) / (s.max() - s.min() + 1e-14)
u_new = u0.min() + s01 * (u0.max() - u0.min())

UV_edit = UV0.copy()
UV_edit[:, 0] = u_new

# Detect global mirror vs local fold-overs
a_ref = signed_double_area_uv(UV0, F)
a_new = signed_double_area_uv(UV_edit, F)
eps = 1e-12 * max(1.0, np.max(np.abs(a_ref)))

flipped_vs_ref = np.where((a_ref * a_new <= 0.0) | (np.abs(a_new) <= eps))[0]

# Local flipps
local_flipped_mask = local_flip_mask(UV_edit, F, eps=eps)
local_flipped = np.where(local_flipped_mask)[0]

print("Faces flagged vs reference:", flipped_vs_ref.size, "/", F.shape[0])
print("True local fold-overs:", local_flipped.size, "/", F.shape[0])

# Plotting
B = igl.barycenter(V, F)
avg = igl.avg_edge_length(V, F) / 2.0

mesh_scalar = tri_polydata(V, F)
mesh_scalar.point_data["phi_edit"] = u_new

# Planar edited UV with local fold-overs in red
UV3 = np.column_stack([UV_edit, np.zeros(V.shape[0])])
mesh_uv = tri_polydata(UV3, F)
face_rgb = np.full((F.shape[0], 3), 0.80, dtype=np.float64)
face_rgb[local_flipped] = np.array([1.0, 0.1, 0.1])
mesh_uv.cell_data["face_rgb"] = face_rgb

# Show constrained faces + guidance field
mesh_field = tri_polydata(V, F)
field_mask = np.ones(F.shape[0], dtype=np.float64)
field_mask[c_faces] = 0.0
mesh_field.cell_data["constraint_mask"] = field_mask

pl = pv.Plotter(shape=(2, 2), window_size=(1700, 1000))
pl.set_background("white")

pl.subplot(0, 0)
pl.add_text("Edited U function + reconstructed gradient", font_size=10)
pl.add_mesh(mesh_scalar, scalars="phi_edit", cmap="turbo", show_edges=False)
plot_lines(pl, B, B + g_rec * avg, "black")

pl.subplot(0, 1)
pl.add_text("Constraint-guided smooth vector field", font_size=10)
pl.add_mesh(mesh_field, scalars="constraint_mask", cmap="coolwarm", show_edges=True, show_scalar_bar=False)
plot_lines(pl, B, B + R * avg, "black")

pl.subplot(1, 0)
pl.add_text("Edited UV map (local fold-overs in red)", font_size=10)
pl.add_mesh(mesh_uv, scalars="face_rgb", rgb=True, show_edges=True, line_width=0.35)
pl.view_xy()

pl.show()

Faces flagged vs reference: 684 / 756
True local fold-overs: 72 / 756


Widget(value='<iframe src="http://localhost:51416/index.html?ui=P_0x1f5f5dce5d0_16&reconnect=auto" class="pyvi…